# 손가락 누름 판정 KNN
웹(collect.html)에서 기록한 관절 좌표로 "이 손가락을 의도적으로 눌렀는가"를 판정하는 모델을 학습한다.
* 학습 데이터 1개 = (박자 1개, 손가락 1개)
* 정답: 악보에서 그 박자에 그 손가락을 누르라고 했으면 1, 아니면 0

## 1. 데이터 불러오기
`session_..._frames.csv`와 `session_..._beats.csv`를 함께 선택한다. 여러 세션을 한 번에 올려도 된다.

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files
    files.upload()
except ImportError:  # Colab이 아닌 곳에서는 현재 폴더의 파일 사용
    pass

sessions = sorted(f[:-len('_frames.csv')] for f in glob.glob('*_frames.csv'))
print(sessions)

## 2. 특징 만들기
MediaPipe 관절 번호: 손목 0, 손가락 끝(TIP) 4·8·12·16·20, 손가락 뿌리 관절(MCP) 2·5·9·13·17

카메라와의 거리에 따라 손 크기가 달라지므로, **손 크기(손목~중지 뿌리 거리)로 나눈 상대 위치**를 쓴다.

$$\text{rel}_y = \frac{y_{\text{끝}} - y_{\text{뿌리}}}{\text{손 크기}}$$

박자 시각 앞 300ms ~ 뒤 200ms 구간에서 손가락마다 세 가지 특징을 계산한다.
* `range_y`: 구간 안에서 rel_y의 최댓값 − 최솟값 (위아래로 얼마나 움직였나)
* `speed_y`: rel_y의 최대 변화 속도 (손 크기/초)
* `range_z`: 깊이 방향 상대 위치의 최댓값 − 최솟값

In [ ]:
FINGERS = ['thumb', 'index', 'middle', 'ring', 'pinky']
TIP = {'thumb': 4, 'index': 8, 'middle': 12, 'ring': 16, 'pinky': 20}
MCP = {'thumb': 2, 'index': 5, 'middle': 9, 'ring': 13, 'pinky': 17}
BEFORE, AFTER = 300, 200  # 박자 기준 구간(ms)
FEATURES = ['range_y', 'speed_y', 'range_z']

def finger_signals(frames, side, f):
    scale = np.hypot(frames[f'{side}_9_x'] - frames[f'{side}_0_x'], frames[f'{side}_9_y'] - frames[f'{side}_0_y'])
    rel_y = (frames[f'{side}_{TIP[f]}_y'] - frames[f'{side}_{MCP[f]}_y']) / scale
    rel_z = (frames[f'{side}_{TIP[f]}_z'] - frames[f'{side}_{MCP[f]}_z']) / scale
    return rel_y.to_numpy(), rel_z.to_numpy()

rows = []
for s in sessions:
    frames = pd.read_csv(f'{s}_frames.csv')
    beats = pd.read_csv(f'{s}_beats.csv')
    beats = beats[beats['discarded'] == 0]  # 스페이스바로 폐기한 박자 제외
    t = frames['t'].to_numpy()
    for side in ['left', 'right']:
        for f in FINGERS:
            y, z = finger_signals(frames, side, f)
            for _, b in beats.iterrows():
                m = (t >= b['t'] - BEFORE) & (t <= b['t'] + AFTER)
                ok = m & ~np.isnan(y)
                if ok.sum() < 3 or ok.sum() < 0.5 * m.sum():  # 손이 절반 이상 안 보이면 제외
                    continue
                yy, zz, tt = y[ok], z[ok], t[ok]
                rows.append({
                    'session': s, 'beat': b['beat'], 'side': side, 'finger': f,
                    'range_y': np.ptp(yy),
                    'speed_y': np.max(np.abs(np.diff(yy) / np.diff(tt))) * 1000,
                    'range_z': np.ptp(zz),
                    'target': int(b[f'{side}_{f}']),
                })

data = pd.DataFrame(rows)
print(data.shape)
data.head()

## 3. 데이터 확인
누름(1)과 안 누름(0)의 개수, 그리고 두 그룹의 특징 분포를 본다.

In [ ]:
print(data['target'].value_counts())

plt.scatter(data.loc[data.target == 0, 'range_y'], data.loc[data.target == 0, 'speed_y'], s=8, label='not pressed (0)')
plt.scatter(data.loc[data.target == 1, 'range_y'], data.loc[data.target == 1, 'speed_y'], s=8, label='pressed (1)')
plt.xlabel('range_y')
plt.ylabel('speed_y')
plt.legend()
plt.show()

### 정확도의 함정
한 박자에 누르는 손가락은 10개 중 0~3개뿐이라 0이 훨씬 많다. **항상 0이라고 답하는 엉터리 모델**도 정확도가 높게 나온다.
그래서 정확도와 함께 두 지표를 본다.
* 재현율: 실제로 누른 것 중 "눌렀다"고 잡아낸 비율 (낮으면 음이 빠짐)
* 정밀도: "눌렀다"고 판정한 것 중 실제로 누른 비율 (낮으면 엉뚱한 음이 남)

In [ ]:
def report(pred, target, name=''):
    acc = np.mean(pred == target)
    recall = np.mean(pred[target == 1] == 1)
    precision = np.mean(target[pred == 1] == 1) if np.any(pred == 1) else 0.0
    print(f'{name:<20} 정확도 {acc:.3f}  재현율 {recall:.3f}  정밀도 {precision:.3f}')
    return acc, recall, precision

y_all = data['target'].to_numpy()
report(np.zeros_like(y_all), y_all, '항상 0')

## 4. 훈련 세트와 테스트 세트
0과 1의 비율이 양쪽에 똑같이 들어가도록 `stratify`를 쓴다.

In [ ]:
from sklearn.model_selection import train_test_split

X = data[FEATURES].to_numpy()
y = data['target'].to_numpy()
train_input, test_input, train_target, test_target, train_idx, test_idx = train_test_split(
    X, y, np.arange(len(y)), stratify=y, random_state=42)
print(train_input.shape, test_input.shape)

## 5. 비교 기준: 규칙 판정
"range_y가 기준값보다 크면 누름"이라는 단순 규칙. 기준값은 **훈련 세트에서** 재현율과 정밀도의 평균이 가장 높은 값으로 정한다.

In [ ]:
def rule_predict(X, th):
    return (X[:, 0] > th).astype(int)

def rule_score(th):
    p, t = rule_predict(train_input, th), train_target
    recall = np.mean(p[t == 1] == 1)
    precision = np.mean(t[p == 1] == 1) if np.any(p == 1) else 0.0
    return (recall + precision) / 2

thresholds = np.linspace(0, np.percentile(train_input[:, 0], 99), 200)
best_th = max(thresholds, key=rule_score)
print('기준값', best_th)
report(rule_predict(test_input, best_th), test_target, '규칙')

## 6. KNN: 스케일링 전과 후

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

kn = KNeighborsClassifier()
kn.fit(train_input, train_target)
report(kn.predict(test_input), test_target, 'KNN (스케일링 전)')

mean = np.mean(train_input, axis=0)
std = np.std(train_input, axis=0)
train_scaled = (train_input - mean) / std
test_scaled = (test_input - mean) / std  # 테스트 세트도 훈련 세트의 평균, 표준편차로 변환

kn.fit(train_scaled, train_target)
report(kn.predict(test_scaled), test_target, 'KNN (스케일링 후)')

## 7. 이웃 개수 k 정하기

In [ ]:
ks = range(1, 26, 2)
results = []
for k in ks:
    kn = KNeighborsClassifier(n_neighbors=k)
    kn.fit(train_scaled, train_target)
    results.append(report(kn.predict(test_scaled), test_target, f'k={k}'))

results = np.array(results)
plt.plot(ks, results[:, 0], label='accuracy')
plt.plot(ks, results[:, 1], label='recall')
plt.plot(ks, results[:, 2], label='precision')
plt.xlabel('k')
plt.legend()
plt.show()

## 8. 손가락별 성능
어떤 손가락에서 많이 틀리는지 본다. (약지처럼 옆 손가락과 같이 움직이는 손가락이 어려울 것으로 예상)

In [ ]:
K = 5  # 7번 결과를 보고 고른 k
kn = KNeighborsClassifier(n_neighbors=K)
kn.fit(train_scaled, train_target)
test_data = data.iloc[test_idx].copy()
test_data['pred'] = kn.predict(test_scaled)

for (side, f), g in test_data.groupby(['side', 'finger']):
    report(g['pred'].to_numpy(), g['target'].to_numpy(), f'{side} {f}')

## 9. 어떤 특징이 중요한가
특징을 하나씩 빼고 학습했을 때 성능이 얼마나 떨어지는지 비교한다.

In [ ]:
for drop in FEATURES:
    cols = [i for i, name in enumerate(FEATURES) if name != drop]
    kn = KNeighborsClassifier(n_neighbors=K)
    kn.fit(train_scaled[:, cols], train_target)
    report(kn.predict(test_scaled[:, cols]), test_target, f'{drop} 제외')